# 08 — Teacher warmup training (ATOMIC + SWOW)

SFT sul dataset misto. Salva in `checkpoints/teacher_warmup_atomic_swow/final/`.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
OUT_DIR = PROJECT_ROOT / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
train_path = PROC_DIR / 'sft_teacher_warmup_train.jsonl'
val_path = PROC_DIR / 'sft_teacher_warmup_val.jsonl'
train_path, val_path

(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/sft_teacher_warmup_train.jsonl'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/sft_teacher_warmup_val.jsonl'))

In [2]:
TEACHER_MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'
RUN_NAME = 'teacher_warmup_atomic_swow'
OUTPUT_DIR = OUT_DIR / RUN_NAME
OUTPUT_DIR

WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/checkpoints/teacher_warmup_atomic_swow')

In [3]:
import json
from datasets import Dataset
def read_jsonl(p):
    rows=[]
    with open(p,'r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows
train_ds = Dataset.from_list(read_jsonl(train_path))
val_ds = Dataset.from_list(read_jsonl(val_path))
train_ds, val_ds

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['id', 'split', 'source', 'event', 'relation', 'messages', 'assistant'],
     num_rows: 20000
 }),
 Dataset({
     features: ['id', 'split', 'source', 'event', 'relation', 'messages', 'assistant'],
     num_rows: 5000
 }))

In [4]:
import torch
print(torch.cuda.is_available())
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

model.config.use_cache = False  # IMPORTANT per training/ckpt
model.gradient_checkpointing_disable()
#model.gradient_checkpointing_enable()

True


Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.99s/it]


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [6]:
def format_example(example):
    msgs = example['messages']
    assistant = example['assistant']
    if hasattr(tokenizer, 'apply_chat_template'):
        text = tokenizer.apply_chat_template(
            msgs + [{'role':'assistant','content': assistant}],
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        text = ''
        for m in msgs:
            text += f"{m['role'].upper()}: {m['content']}\n"
        text += f"ASSISTANT: {assistant}"
    return {'text': text}
train_text = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_text = val_ds.map(format_example, remove_columns=val_ds.column_names)
train_text[0]['text'][:600]

Map: 100%|██████████| 5000/5000 [00:00<00:00, 18313.61 examples/s]


'<｜begin▁of▁sentence｜>You are a helpful assistant specialized in commonsense knowledge completion. Given an event and a relation type, generate ONE plausible completion phrase. Return ONLY the completion phrase.<｜User｜>Event: PersonX sees PersonY coming\nRelation: oEffect\nCompletion:<｜Assistant｜>Confronts X<｜end▁of▁sentence｜>'

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding=False,
    )

train_tok = train_text.map(tokenize, batched=True, remove_columns=['text'])
val_tok = val_text.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  #or 16 if Out Of Memory

    learning_rate=2e-4,         
    num_train_epochs=1,
    warmup_ratio=0.03,

    fp16=True,
    optim="paged_adamw_8bit",

    logging_steps=25,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,

    report_to="none",
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)
trainer

Map: 100%|██████████| 5000/5000 [00:00<00:00, 44245.58 examples/s]


In [8]:
trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'final'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'final'))
print('Ready. Uncomment trainer.train() when compute is configured.')

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
500,0.724400,0.763544
1000,0.724400,0.752465
1500,0.663400,0.750081
2000,0.664200,0.751262
2500,0.627000,0.750810


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7bef1729-b763-4c51-b572-0b13bf2c381e)')' thrown while requesting HEAD https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Llama-8B/resolve/main/config.json
Retrying in 1s [Retry 1/5].
c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant pa

Ready. Uncomment trainer.train() when compute is configured.
